2026/01/13  By Li.Jiahao.
Nowadays, asymmetric encryption is widely used in various fields, because of its complexity. Take the RSA2048 as an example, it is impossible to crack the private key for a classical computer since it takes hundreds of years to finish(Basically, it is about breaking down a semiprime). But for qpu, it can be very easy. This article will introduce why qpu can easily break down a semiprime.

# Shor's Algorithm

## 1.The order-finding problem
At first, we will introduce modular arithmetic which is a commonplace in computation science.
$$a \equiv b \pmod{n} \text{(If we divide a by n, the remainder is b.)}$$

For any a semiprime N, we can find two numbers x and y(or r) where $(x,y,r) \in {0,1,...,N-1}$, satisfies:
$$xy \equiv 1 \pmod{N} \ \ (1)$$
or
$$x^r\equiv 1 \pmod{N} \ \ (2)$$
In other words, gcd(x,N)=1, x and N are coprime numbers. We focus on the equation (2) to find a way to solve prime factorization problem. For equation (1),trying to select a number from 0 to N-1 and letting it satisfy gcd(N,x)>1, is difficult, especially when bit number is 2048(Classical computer always do like that). For the other, we can use QFT to find the order r and solve it as(It is easy to find a number x satistying gcd(N,x)=1):
$$(x^{r/2})^2 - 1 \equiv 0 \pmod n$$
$$(x^{r/2} - 1)(x^{r/2} + 1) \equiv 0 \pmod n$$



## 2.Quantum Implementation

### 2.1 Problem statement and connection to phase estimation
For obtaining the order r of a specific number a. We need to build a system to map the order-finding problem into phase estimation problem.
To do this, we define a special matrix $M_a$ which satisties that:
$$M_a |x\rangle  = |xa\rangle \ \ , x \in {0,1,...N-1} \ \ \text{(The '=' denotes modular arithmetic for simplicity) } $$
And we can recognize that the inverse of $M_a$ is $M_a ^{(r-1)}$,because of that:
$$M_{a^{r-1}} M_a = M_a^r = M_{a^r} = M_1 = \mathbb{I}$$
If we do not know what the 'r' exactly is, we have:
$$M_{a^{-1}} M_a  = M_{a^{-1}a} = M_1 = \mathbb{I}$$
So, the operation $M_a$ is deterministic and invertible.That implies that it's described by a permutation matrix, and is therefore unitary.
For the operation, it has N eigenvalues and corresponding N eigenvectors. Let's start simply and identify just one eigenvector:
$$|\psi_0\rangle = \frac{|1\rangle + |a\rangle + \dots + |a^{r-1}\rangle}{\sqrt{r}}$$
And we can get:
$$M_a |\psi_0\rangle = \frac{|a\rangle + \dots + |a^{r-1}\rangle + |a^r\rangle}{\sqrt{r}} = \frac{|a\rangle + \dots + |a^{r-1}\rangle + |1\rangle}{\sqrt{r}} = |\psi_0\rangle$$
Let's introduce a phase coefficient into the eigenvector,and we have:
$$|\psi_1\rangle = \frac{|1\rangle + \omega_r^{-1}|a\rangle + \dots + \omega_r^{-(r-1)}|a^{r-1}\rangle}{\sqrt{r}} \ \ , \ \ |\psi_1\rangle = \frac{1}{\sqrt{r}} \sum_{k=0}^{r-1} \omega_r^{-k} |a^k\rangle$$
$$M_a |\psi_1\rangle = \sum_{k=0}^{r-1} \omega_r^{-k} M_a |a^k\rangle = \sum_{k=0}^{r-1} \omega_r^{-k} |a^{k+1}\rangle = \sum_{k=1}^{r} \omega_r^{-(k-1)} |a^k\rangle = \omega_r \sum_{k=1}^{r} \omega_r^{-k} |a^k\rangle = \omega_r |\psi_1 \rangle$$

Through that, we can get another eigenvalue of $\omega_r$.
We can generalize this, for an arbitrary eigenvalue taking the form as $|\psi_j\rangle$,we have:
$$M_a|\psi_j\rangle = \omega_r^j|\psi_j\rangle$$
Although we do not know the value of the order r, the eigenvector $\omega_r^j$ contains precisely the information we need. 

### 2.2 Order finding circuit 

Section 2.1 established the connection between order finding and phase estimation. Here, we will explain the implementation of the order finding circuit in detail.
First, the number N must be encode. We use log(N) bits  to represent it. For a randomly selected number a, we first compute its greatest common divisor with N, if gcd(a,N)>1 , we have successfully found a non-trivial factor of N, and the algorithm can terminate. Otherwise, we proceed to construct a circuit that implements the unitary operator $M_a$,defined as follows:
$$M_a |x\rangle = 
\begin{cases} 
      |ax \ (\text{mod } N)\rangle & 0 \le x < N \\
      |x\rangle & N \le x < 2^n 
\end{cases}$$
As previous discussed, techniques exist to implement classical computation reversibly and uncompute any auxiliary qubits(garbage recycle). We define the f(x) as follows:
$$ f(x) = 
\begin{cases} 
      ax \ (\text{mod } N) & 0 \le x < N \\
      x & N \le x < 2^n 
\end{cases}$$
And after cleaning garbage and applying swap gates, system evolves into:
$$|x\rangle|0^k\rangle|y\rangle \mapsto |x\rangle|0^k\rangle|y \oplus f_{a}(x)\rangle \ \ (1)$$
Let's initialize the state y to state zero, and no longer pay attention to the auxiliary state $|0^k\rangle$, we get state $|x\rangle|f_{a}(x)\rangle$.
Because of the reversibility of $M_a$, we build the circuit described as:
$$|x\rangle|y\rangle \mapsto |x\rangle|y \oplus f_{a^{-1}}(x)\rangle \ \ (2)$$
We recycle garbage again and get:
$$|f_a(x)\rangle |x \oplus f_{a^{-1}}(f_a(x))\rangle = |f_a(x)\rangle |0^n\rangle$$
So far, we have successfully constructed the circuit of $M_a$ with the cost of $O(n^2)$ .

### 2.3 Eigenvalue analysis
Next, Let's analyze the outcome derives from the operation $M_a$. For the circuit $M_a$ ,we can obtain the corresponding outcome:$\omega_r = e^{2\pi i \frac{1}{r}}.$ If we run the QPE we can get a approximately value of phase which's reciprocal is the order 'r'. It returns to the accuracy problem about QPE we used to introduce. Let's extend the theory about the "Bounding the probabilities".

Since $r$ is an integer, intuitively, we expect $1/r$ to be close to its neighbors $1/(r+1)$ and $1/(r-1)$. By the squeeze effect, as $r$ increases, the gap between $1/r$ and $1/(r+1)$ becomes smaller. If we can distinguish this gap, we can similarly distinguish the gap between $1/r$ and $1/(r-1)$.

We need to ensure that the precision of phase estimation is sufficient compared to $1/r$. So, we define that: 
$$\frac{1}{r} - \frac{1}{r + 1} = \frac{1}{r(r + 1)}, \\

\left| \frac{y}{2^m} - \frac{1}{r} \right| < \frac{1}{2r(r + 1)}$$

Suppose that
$$\frac{y}{2^m} = \frac{1}{r} + \varepsilon$$
and
$$\left| \varepsilon \right| < \frac{1}{2r(r + 1)}.$$
When we take the reciprocal and obtain:
$$\frac{2^m}{y} = \frac{1}{\frac{1}{r} + \varepsilon} = \frac{r}{1 + \varepsilon r} = r - \frac{\varepsilon r^2}{1 + \varepsilon r}.$$
By maximizing in the numerator minimizing in the denominator, we can bonud how far away we are from r as follows:
$$\left| \frac{\varepsilon r^2}{1 + \varepsilon r} \right| \le \frac{\frac{r^2}{2r(r+1)}}{1 - \frac{r}{2r(r+1)}} = \frac{r}{2r + 1} < \frac{1}{2}$$
We are less than 1/2 from r, so we can get r when we do rounding.
Unfortunately, we do not know the value of $r$, so we cannot determine the exact accuracy required. What we do know is the condition $r < N$. According to the Continued Fractions Theorem, if we want $\frac{2^m}{y}$ to be as close as possible to $1/r$, the error must be less than $\frac{1}{2r^2}$.
So, we get 
$$\left| \frac{y}{2^m} - \frac{1}{r} \right| \le \frac{1}{2N^2}$$
And we need m qubits to run the QPE where $m=2lg(N)+1$.